# Policy Events and Emissions

### Imports and preprocessing

In [6]:
import sys
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path

ROOT_DIR = Path.cwd().parent
Tables_DIR = ROOT_DIR / "data/tables"

# helper that saves each graph + summary so src/make_pdf.py can build the PDF
sys.path.insert(0, str(ROOT_DIR / "src"))
from report import save_report

event_countriesdf = pd.read_csv(Tables_DIR / "event_countries.csv")
eventsdf          = pd.read_csv(Tables_DIR / "events.csv")
country_yearlydf  = pd.read_csv(Tables_DIR / "country_yearly.csv")
global_yearlydf   = pd.read_csv(Tables_DIR / "global_yearly.csv")
coveragedf        = pd.read_csv(Tables_DIR / "data_coverage.csv")

for name, d in [("country_yearly", country_yearlydf), ("global_yearly", global_yearlydf),
                ("events", eventsdf), ("event_countries", event_countriesdf)]:
    print(name, list(d.columns))

# ---------- column auto-detection (edit candidates if needed) ----------
def col(df, *names):
    for n in names:
        if n in df.columns:
            return n
    raise KeyError(f"None of {names} found in {list(df.columns)}")

C_COUNTRY = col(country_yearlydf, "country", "country_name", "entity")
C_ISO     = col(country_yearlydf, "iso_code", "iso3", "country_code", "iso")
C_YEAR    = col(country_yearlydf, "year")
C_CO2     = col(country_yearlydf, "co2", "co2_mt", "annual_co2")
C_PC      = col(country_yearlydf, "co2_per_capita", "co2_pc", "co2_percap")
C_PM      = col(country_yearlydf, "pm25", "pm2_5", "pm25_exposure")

G_YEAR = col(global_yearlydf, "year")
G_TEMP = col(global_yearlydf, "temp_anomaly", "temperature_anomaly", "temp_anom", "anomaly", "j_d", "temp")

E_ID   = col(eventsdf, "event_id", "id")
E_YEAR = col(eventsdf, "year", "event_year")
E_NAME = col(eventsdf, "event_name", "name", "title", "event")
EC_ID  = col(event_countriesdf, "event_id", "id")
EC_ISO = col(event_countriesdf, "iso_code", "iso3", "country_code", "iso")

# real countries only (drop OWID aggregates like World, Asia, EU...)
is_country = (country_yearlydf[C_ISO].notna()
              & ~country_yearlydf[C_ISO].astype(str).str.startswith("OWID"))
cdf = country_yearlydf[is_country].copy()

# ---------- chart style ----------
NAVY, BLUE, RED, DARK = "#2c3e5e", "#4a78a8", "#e05252", "#2b2b2b"
FILL, GRID = "rgba(74,120,168,0.10)", "#e6eaf2"

def style(fig, title, subtitle, xt, yt, h=450, ylim=None):
    fig.update_layout(
        title=dict(text=f"<b>{title}</b><br><sup>{subtitle}</sup>",
                   x=0.02, xanchor="left", font=dict(size=20, color=NAVY)),
        template="plotly_white", width=1090, height=h,
        margin=dict(l=75, r=45, t=95, b=95),
        font=dict(color=NAVY, size=13),
        plot_bgcolor="white", paper_bgcolor="white",
        legend=dict(orientation="h", x=0.5, xanchor="center", y=-0.25),
    )
    fig.update_xaxes(title=xt, gridcolor=GRID, zeroline=False)
    fig.update_yaxes(title=yt, gridcolor=GRID, zeroline=False, range=ylim)
    return fig

def dot(x, y, label, color, name, pos="top left", showlegend=True):
    return go.Scatter(x=[x], y=[y], mode="markers+text", text=[label],
                      textposition=pos, name=name, showlegend=showlegend,
                      marker=dict(size=11, color=color), textfont=dict(color=NAVY))

def world_series(value_col):
    """World total per year = sum over real countries."""
    return cdf.groupby(C_YEAR)[value_col].sum(min_count=1)

country_yearly ['iso_code', 'country', 'year', 'population', 'gdp', 'co2_mt', 'co2_per_capita', 'cumulative_co2', 'share_global_co2', 'coal_co2', 'oil_co2', 'gas_co2', 'cement_co2', 'flaring_co2', 'methane', 'nitrous_oxide', 'total_ghg', 'pm25_exposure', 'co2_yoy_pct', 'rank_total_co2', 'rank_per_capita', 'co2_per_gdp', 'gdp_change_10yr_pct', 'co2_change_10yr_pct', 'decoupled_flag']
global_yearly ['year', 'decade', 'is_complete_year', 'co2_ppm', 'co2_mlo_ppm', 'ch4_ppb', 'n2o_ppb', 'temp_anomaly', 'world_co2_mt', 'co2_mlo_yoy_change', 'co2_pct_index', 'ch4_pct_index', 'n2o_pct_index', 'temp_yoy_change', 'co2_mlo_detrended', 'temp_detrended']
events ['event_id', 'event_name', 'year', 'event_type', 'source_note']
event_countries ['event_id', 'iso_code', 'role']


In [7]:
TOPIC = "policy_events"

### Q1. What happens to emissions after major energy-policy changes?

In [8]:
ev = eventsdf.merge(event_countriesdf, left_on=E_ID, right_on=EC_ID)
iso_p = cdf.pivot_table(index=C_ISO, columns=C_YEAR, values=C_CO2)

rows = []
for _, r in ev.iterrows():
    iso, y = r[EC_ISO], int(r[E_YEAR])
    if iso not in iso_p.index:
        continue
    s = iso_p.loc[iso]
    before = s.loc[y - 3:y - 1].mean()
    after = s.loc[y + 1:y + 3].mean()
    if pd.notna(before) and pd.notna(after) and before > 0:
        rows.append((r[E_NAME], y, (after / before - 1) * 100))

res = (pd.DataFrame(rows, columns=["event", "year", "pct"])
       .groupby(["event", "year"], as_index=False).pct.mean().sort_values("pct"))
res["label"] = res["event"].str.slice(0, 40) + " (" + res["year"].astype(str) + ")"
if res.empty:
    raise ValueError("No events could be matched to country emissions data.")

# ---- graph ----
fig = go.Figure(go.Bar(x=res["pct"], y=res["label"], orientation="h",
                       marker_color=[RED if v < 0 else BLUE for v in res["pct"]],
                       text=[f"{v:+.1f}%" for v in res["pct"]], textposition="outside",
                       name="Avg change in emissions"))
style(fig, "Emissions After Major Energy-Policy Events",
      "Average CO2 in the 3 years after vs 3 years before (affected countries)",
      "Change in CO2 (%)", "", h=max(450, 40 * len(res) + 160))
fig.show()

# ---- summary ----
summary = f"""
Of {len(res)} events analysed, {int((res.pct < 0).sum())} were followed by lower average emissions in the
countries involved. The largest decline followed "{res.iloc[0]['event']}" ({res.iloc[0]['year']}, {res.iloc[0].pct:.1f}%),
while "{res.iloc[-1]['event']}" ({res.iloc[-1]['year']}) saw a {res.iloc[-1].pct:+.1f}% change.
This is a before/after comparison, not proof of causation, since the wider economy and other policies move at the same time.
""".strip()
print(summary)
save_report(TOPIC, "q1", "What happens to emissions after major energy-policy changes?", fig, summary)

Of 4 events analysed, 0 were followed by lower average emissions in the
countries involved. The largest decline followed "Global financial crisis (2008-09)" (2008, 3.0%),
while "China air-pollution action plan" (2013) saw a +12.8% change.
This is a before/after comparison, not proof of causation, since the wider economy and other policies move at the same time.
[report] saved policy_events/q1 (1 graph(s))


### Q2. Did the 2020 COVID drop in CO2 show up, how large was it, and how quickly did emissions rebound?

In [9]:
w = world_series(C_CO2).loc[2015:2024] / 1000          # Gt CO2
drop_pct = (w[2020] / w[2019] - 1) * 100
drop_gt = w[2019] - w[2020]
reb_pct = (w[2021] / w[2020] - 1) * 100
back_year = next((y for y in w.index if y > 2020 and w[y] >= w[2019]), None)

# ---- graph ----
fig = go.Figure()
fig.add_trace(go.Scatter(x=w.index, y=w.values, mode="lines", name="Global CO2 emissions",
                         line=dict(color=BLUE, width=3), fill="tozeroy", fillcolor=FILL))
fig.add_trace(dot(2019, w[2019], f"2019<br>{w[2019]:.1f} Gt", RED, "2019 peak"))
fig.add_trace(dot(2020, w[2020], f"2020<br>{w[2020]:.1f} Gt", RED, "2020 COVID low", pos="bottom center"))
fig.add_trace(dot(w.index[-1], w.iloc[-1], f"{w.index[-1]}<br>{w.iloc[-1]:.1f} Gt", DARK,
                  "Latest", pos="top center", showlegend=False))
style(fig, "Global CO2 Emissions Around COVID-19, 2015-2024",
      f"{drop_pct:.1f}% drop in 2020, then {reb_pct:+.1f}% in 2021",
      "Year", "Emissions (Gt CO2)", ylim=[w.min() * 0.9, w.max() * 1.06])
fig.show()

# ---- summary ----
back_txt = f"returned to the 2019 level in {back_year}" if back_year else "had not yet exceeded the 2019 level by 2024"
summary = f"""
Global CO2 emissions fell from {w[2019]:.2f} Gt in 2019 to {w[2020]:.2f} Gt in 2020, a drop of
{drop_gt:.2f} Gt ({drop_pct:.1f}%), the sharpest single-year fall in the recent record. Emissions
rebounded {reb_pct:.1f}% in 2021 and {back_txt}.
The drop was therefore real but temporary: it reflected lockdown-related demand, not structural change.
""".strip()
print(summary)
save_report(TOPIC, "q2",
            "Did the 2020 COVID drop in CO2 show up, how large was it, and how quickly did emissions rebound?",
            fig, summary)

Global CO2 emissions fell from 35.84 Gt in 2019 to 34.31 Gt in 2020, a drop of
1.53 Gt (-4.3%), the sharpest single-year fall in the recent record. Emissions
rebounded 4.9% in 2021 and returned to the 2019 level in 2021.
The drop was therefore real but temporary: it reflected lockdown-related demand, not structural change.
[report] saved policy_events/q2 (1 graph(s))


### Q3. Which countries saw the biggest CO2 drop after 2008-09 and recovered fastest?

In [10]:
p = cdf.pivot_table(index=C_COUNTRY, columns=C_YEAR, values=C_CO2)
big = p[p[2008] >= 100]                                   # emitters >= 100 Mt only
chg = ((big[2009] / big[2008] - 1) * 100).sort_values().head(12)

def recovery_year(c):
    s = p.loc[c]
    return next((y for y in s.index if y > 2009 and s[y] >= s[2008]), None)

rec = {c: recovery_year(c) for c in chg.index}
lbl = [f"{v:.1f}%  (recovered {rec[c] if rec[c] else 'not yet'})" for c, v in chg.items()]

# ---- graph ----
fig = go.Figure(go.Bar(x=chg.values[::-1], y=chg.index[::-1], orientation="h",
                       marker_color=BLUE, text=lbl[::-1], textposition="outside",
                       name="Change 2008 to 2009"))
fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", marker=dict(size=11, color=RED),
                         name="Label shows year emissions regained 2008 level"))
style(fig, "Biggest CO2 Drops After the 2008-09 Financial Crisis",
      "Countries emitting 100+ Mt in 2008; % change 2008 to 2009", "Change in CO2 (%)", "", h=520)
fig.update_xaxes(range=[chg.min() * 1.6, 2])
fig.show()

# ---- summary ----
fast = sorted([(c, y) for c, y in rec.items() if y], key=lambda t: t[1])[:3]
steepest = ", ".join(f"{c} ({chg[c]:.1f}%)" for c in chg.index[:3])
fast_txt = ("The fastest to regain their 2008 emissions were " + ", ".join(f"{c} ({y})" for c, y in fast) + "."
            if fast else "None of the top-drop countries regained their 2008 level in the data.")
summary = f"""
Among large emitters, the steepest falls were {steepest}.
{fast_txt}
Countries with the deepest drops did not always recover fastest; recovery depended on whether the
fall was cyclical (demand) or structural (fuel switching, deindustrialisation).
""".strip()
print(summary)
save_report(TOPIC, "q3", "Which countries saw the biggest CO2 drop after 2008-09 and recovered fastest?",
            fig, summary)

Among large emitters, the steepest falls were Romania (-17.9%), Uzbekistan (-14.9%), Ukraine (-14.8%).
The fastest to regain their 2008 emissions were Russia (2011), Uzbekistan (2024).
Countries with the deepest drops did not always recover fastest; recovery depended on whether the
fall was cyclical (demand) or structural (fuel switching, deindustrialisation).
[report] saved policy_events/q3 (1 graph(s))
